# FluxCompute — Full Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fluxcompute/fluxcompute-sdk/blob/main/examples/full_walkthrough.ipynb)

A guided tour of the FluxCompute SDK. Start with [`quickstart.ipynb`](quickstart.ipynb) if you just want the 5-minute version.

**You'll learn to:**
1. Make auto-routed calls and read the routing/cost metadata
2. See routing decisions across easy / medium / hard tiers
3. Measure aggregate savings
4. Migrate from the Anthropic SDK (drop-in)
5. Use **multi-turn sessions** with context-aware routing
6. **Stream** responses
7. Record an **execution graph** of a multi-step task — and **resume** a failed step

**Prerequisite:** an Anthropic API key ([console.anthropic.com](https://console.anthropic.com/)). Calls are real.

## 1 · Setup

In [ ]:
%pip install -q "fluxcompute>=0.3.0" matplotlib pandas

In [ ]:
import os, getpass

# FluxCompute calls Anthropic for real, so you need an Anthropic API key.
# Get one at https://console.anthropic.com/ . It is read via getpass — never
# hardcode keys in a notebook you might share.
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key (sk-ant-...): ")

from fluxcompute import FluxClient

# Drop-in replacement for anthropic.AsyncAnthropic. `baseline_model` defaults to
# the priciest tier (Opus) — that's what your savings are measured against.
client = FluxClient(anthropic_key=os.environ["ANTHROPIC_API_KEY"])
print("FluxClient ready.")

## 2 · First auto-routed call

In [ ]:
# NOTE: the SDK is async. In a Jupyter cell you can `await` directly;
# in a plain .py script, wrap calls in asyncio.run(...).
resp = await client.messages.create(
    model="auto",                                   # let FluxCompute pick the model
    max_tokens=512,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
)

print(resp.text, "\n")

m = resp.fluxcompute
pct = (m.savings_usd / m.baseline_cost_usd * 100) if m.baseline_cost_usd else 0
print(f"Routed to : {m.model_selected}  (difficulty: {m.difficulty_label}, score {m.difficulty_score})")
print(f"You paid  : ${m.cost_usd:.6f}")
print(f"All-Opus  : ${m.baseline_cost_usd:.6f}  (baseline: {m.baseline_model})")
print(f"Saved     : ${m.savings_usd:.6f}  ({pct:.0f}% cheaper) — for a question Opus is overkill for")

## 3 · Routing across difficulty tiers

In [ ]:
import pandas as pd

samples = [
    "What is 15% of 200?",                                                           # easy
    "Explain the difference between supervised and unsupervised learning.",          # medium
    ("Design a distributed rate limiter across 50 instances with Redis. Analyze "    # hard
     "the trade-offs between token bucket and sliding window, justify your choice, "
     "and prove the algorithm is correct under network partition."),
]

def short_model(name: str) -> str:
    """Drop the vendor prefix and any trailing date stamp for a readable table."""
    name = name.replace("claude-", "")
    parts = name.rsplit("-", 1)
    # only an 8-digit date stamp is a suffix; "-6" in sonnet-4-6 is part of
    # the version and stripping it renames the model
    is_date = len(parts) == 2 and len(parts[1]) == 8 and parts[1].isdigit()
    return parts[0] if is_date else name

rows = []
for q in samples:
    r = await client.messages.create(model="auto", max_tokens=512,
                                      messages=[{"role": "user", "content": q}])
    m = r.fluxcompute
    rows.append({
        "query": (q[:45] + "\u2026") if len(q) > 45 else q,
        "difficulty": m.difficulty_label,
        "routed_to": short_model(m.model_selected),
        "cost_usd": round(m.cost_usd, 6),
        "all_opus_usd": round(m.baseline_cost_usd, 6),
        "saved_usd": round(m.savings_usd, 6),
    })

pd.DataFrame(rows)

## 4 · Aggregate savings

In [ ]:
import matplotlib.pyplot as plt

# A realistic mix: mostly easy/medium traffic, with a couple of genuinely hard asks.
bank = [
    "What does HTTP stand for?",
    "Translate 'good morning' to Japanese.",
    "Who wrote Pride and Prejudice?",
    "What is 240 / 8?",
    "Summarize the CAP theorem in two sentences.",
    "Compare REST and GraphQL and say when to use each.",
    "Write a Python function that checks if a string is a palindrome.",
    "Explain how a transformer attention head works.",
    ("Implement an LFU cache with O(1) get/put in Python. Analyze the trade-offs "
     "against an LRU design, prove the complexity bound, and justify the tie-breaking rule."),
    ("Debug why adding a composite index to a 500M-row Postgres table under 50k "
     "inserts/sec deadlocks. Analyze the lock ordering, prove the fix removes the "
     "cycle, and evaluate the trade-offs of doing it online."),
]

metas = []
for q in bank:
    r = await client.messages.create(model="auto", max_tokens=512,
                                      messages=[{"role": "user", "content": q}])
    metas.append(r.fluxcompute)

total_actual   = sum(m.cost_usd for m in metas)
total_baseline = sum(m.baseline_cost_usd for m in metas)
total_saved    = sum(m.savings_usd for m in metas)
pct = (total_saved / total_baseline * 100) if total_baseline else 0

print(f"{len(metas)} queries")
print(f"All-Opus baseline : ${total_baseline:.4f}")
print(f"FluxCompute (auto): ${total_actual:.4f}")
print(f"Saved             : ${total_saved:.4f}   ({pct:.0f}% reduction)")
print()
print("Below the 80%/40% per-query figures because the hard queries route to")
print("Opus — the baseline itself — so they save nothing by construction.")
print("Your number depends on how much of your traffic needs a frontier model.")

plt.figure(figsize=(5, 3.2))
bars = plt.bar(["All-Opus\n(baseline)", "FluxCompute\n(auto)"],
               [total_baseline, total_actual], color=["#cbd5e1", "#3b82f6"])
plt.ylabel("Total cost (USD)")
plt.title(f"{pct:.0f}% cheaper on this workload")
for b, v in zip(bars, [total_baseline, total_actual]):
    plt.text(b.get_x() + b.get_width() / 2, v, f"${v:.4f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 5 · Drop-in compatibility

In [ ]:
# FluxResponse behaves like the raw provider response, so your existing code keeps working:
print("model :", resp.model)
print("usage :", resp.usage)            # {'input_tokens': ..., 'output_tokens': ...}
print("text  :", resp.text[:60], "...")

# Migrating from the Anthropic SDK is a 2-line change:
#
#   - from anthropic import AsyncAnthropic
#   - client = AsyncAnthropic(api_key=KEY)
#   - resp = await client.messages.create(model="claude-opus-4-8", max_tokens=512, messages=msgs)
#   - text = resp.content[0].text
#   + from fluxcompute import FluxClient
#   + client = FluxClient(anthropic_key=KEY)
#   + resp = await client.messages.create(model="auto", max_tokens=512, messages=msgs)
#   + text = resp.text          # resp.content / resp.usage / resp.model also work

## 6 · Multi-turn sessions (context-aware routing)

Real agents are conversations, not one-shots. Give a `session_id` and FluxCompute keeps the thread for you — you send only the new message, and the classifier judges difficulty **in context**. Watch the same session escalate from Haiku to a stronger model as it gets into real code.

In [ ]:
# Pass a session_id and FluxCompute tracks the conversation for you — you only
# send the *new* user message each turn. The classifier sees the full thread, so a
# question that's trivial in isolation can route UP once the session gets complex.
session_id = "demo-coding-session"

turns = [
    "What's a good way to store user sessions?",
    "Show me a Redis-backed implementation in Python.",
    ("Now analyze the trade-offs of this design under a network partition, "
     "prove whether sessions can be lost, and justify a fix."),
]

for t in turns:
    r = await client.messages.create(
        model="auto",
        max_tokens=512,
        session_id=session_id,
        messages=[{"role": "user", "content": t}],
    )
    m = r.fluxcompute
    print(f"{t[:52]:52s} -> {m.difficulty_label:6s} ({m.model_selected.replace('claude-','')})")

## 7 · Streaming

Same routing, token-by-token output. The cost/routing metadata is available on the stream after iteration completes.

In [ ]:
# Streaming uses an async context manager. Iterate for text deltas; after the loop,
# stream.fluxcompute holds the routing + cost metadata.
async with client.messages.stream(
    model="auto",
    max_tokens=256,
    messages=[{"role": "user", "content": "Write a haiku about routing LLM queries to the right model."}],
) as stream:
    async for chunk in stream:
        print(chunk.text, end="", flush=True)

m = stream.fluxcompute
print(f"\n\n[{m.model_selected} · {m.difficulty_label} · saved ${m.savings_usd:.6f}]")

## 8 · Execution graphs — and resuming a failed step

Wrap multi-step work in a `task()` scope and the SDK records a DAG of everything
inside it — LLM calls and your own steps, auto-parented, with status, timings,
tokens, and cost. It works in any framework, with no integration code.

If a step fails, `client.resume()` rebuilds minimal context from what already
succeeded and re-runs only the failed step — free, in-process, no network call
beyond the retry itself. The retry lands in the *same* graph, linked to the
node it replaces.

In [ ]:
with client.task("competitive-research") as t:
    await client.messages.create(
        model="auto", max_tokens=256,
        messages=[{"role": "user", "content": "Name one competitor to Redis."}],
    )
    with client.step("record-pricing") as s:
        s.set_output("competitor A: $10/mo")

    # Simulate a flaky downstream call failing mid-task.
    try:
        with client.step("check-competitor-uptime"):
            raise TimeoutError("status API timed out")
    except TimeoutError:
        pass

print("Before resume:")
graph = client.get_task_graph(t.task_id)
for node in graph.in_order():
    print(f"  {node.node_type:10s} {node.name:24s} {node.status:9s} ${node.cost_usd:.6f}")

response = await client.resume(t.task_id)

print("\nAfter resume:")
graph = client.get_task_graph(t.task_id)
for node in graph.in_order():
    linked = f" -> depends_on {node.depends_on}" if node.depends_on else ""
    print(f"  {node.node_type:10s} {node.name:24s} {node.status:9s} ${node.cost_usd:.6f}{linked}")

print(f"\nresumed answer: {response.text[:80]!r}")

## Recap & next steps

You've seen auto-routing, the savings story, drop-in migration, sessions,
streaming, and execution graphs.

- **Production:** swap `FluxClient` in wherever you construct your provider client.
- **Dashboard** (invite-only early access): pass `fluxcompute_key="flx_..."` to
  stream routing and cost metrics. Response text stays on your machine unless
  you also pass `content_capture=True` — see the Telemetry & privacy section of
  the [README](../README.md) and
  [docs/telemetry-contract.md](../docs/telemetry-contract.md). Without a
  reachable endpoint the SDK simply doesn't report; request access at
  [fluxcompute.dev](https://fluxcompute.dev).
- **Recovery:** `client.resume()` re-runs a failed step from reconstructed
  minimal context, as you saw above — free and in-process, for as long as the
  process that ran the task is still alive. Resuming a task whose process has
  since exited means rebuilding its graph from a persisted copy, which needs a
  recovery plugin; see [fluxcompute.dev](https://fluxcompute.dev).

Clean up:

In [ ]:
# Flushes telemetry and closes the underlying provider client.
await client.close()
print("done.")